# PMC Chunks Token 分析与 JSON 导出

逐批读取 `batches_full/` 目录，不把全量数据加载进内存。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BATCH_DIR = Path(r'd:\Rag-Med\pipeline_output\batches_full')
batch_files = sorted(BATCH_DIR.glob('batch_*.parquet'))
print(f'批次文件数：{len(batch_files):,}')

## Cell 1 — Token 总量分析

In [ ]:
# ── 逐批累积统计，内存占用 ≈ 单批大小 ──────────────────────────────────
READ_COLS = ['token_count', 'chunk_type', 'imrad_type', 'split_strategy']

total_chunks   = 0
total_tokens   = 0
over_512       = 0

# 分组计数器
by_chunk_type  : dict[str, dict] = {}   # {type: {chunks, tokens}}
by_imrad       : dict[str, dict] = {}
by_strategy    : dict[str, dict] = {}

# Token 直方图 bins
BINS   = [0, 50, 100, 200, 300, 400, 512, 600, 800, 10000]
LABELS = ['<50','50-99','100-199','200-299','300-399','400-511','512-599','600-799','>=800']
hist   = {l: 0 for l in LABELS}

# 用于 p90/p95 近似（reservoir sampling 100w 样本）
RESERVOIR_SIZE = 1_000_000
reservoir: list[int] = []
rng = np.random.default_rng(42)

LOG_EVERY = 200

for i, bf in enumerate(batch_files):
    df = pd.read_parquet(bf, columns=READ_COLS)
    tc = df['token_count'].dropna().astype(int)

    n   = len(tc)
    s   = int(tc.sum())
    total_chunks += n
    total_tokens += s
    over_512     += int((tc > 512).sum())

    # 直方图
    for j, label in enumerate(LABELS):
        lo, hi = BINS[j], BINS[j+1]
        hist[label] += int(((tc >= lo) & (tc < hi)).sum())

    # 分组累积
    for col, counter in [('chunk_type', by_chunk_type),
                          ('imrad_type',  by_imrad),
                          ('split_strategy', by_strategy)]:
        if col in df.columns:
            grp = df.groupby(col)['token_count'].agg(['count','sum']).dropna()
            for k, row in grp.iterrows():
                if k not in counter:
                    counter[k] = {'chunks': 0, 'tokens': 0}
                counter[k]['chunks'] += int(row['count'])
                counter[k]['tokens'] += int(row['sum'])

    # Reservoir sampling for percentile estimation
    vals = tc.tolist()
    if len(reservoir) < RESERVOIR_SIZE:
        reservoir.extend(vals)
        if len(reservoir) > RESERVOIR_SIZE:
            reservoir = reservoir[:RESERVOIR_SIZE]
    else:
        # replace with probability RESERVOIR_SIZE / total_chunks
        for v in vals:
            idx = rng.integers(0, total_chunks)
            if idx < RESERVOIR_SIZE:
                reservoir[idx] = v

    if (i + 1) % LOG_EVERY == 0 or (i + 1) == len(batch_files):
        pct = (i + 1) / len(batch_files) * 100
        print(f'  [{i+1:>4}/{len(batch_files)}  {pct:.0f}%]  '
              f'chunks={total_chunks:,}  tokens={total_tokens:,}')

# ── 输出结果 ─────────────────────────────────────────────────────────────
rsv = np.array(reservoir)
print(f'\n{"="*60}')
print(f'  Token 总量分析')
print(f'{"="*60}')
print(f'  总 chunk 数       : {total_chunks:>15,}')
print(f'  总 token 数       : {total_tokens:>15,}')
print(f'  超 512 tokens     : {over_512:>15,}  ({over_512/total_chunks*100:.2f}%)')
print(f'  均值 (approx)     : {total_tokens/total_chunks:>15.1f}')
print(f'  p90  (reservoir)  : {np.percentile(rsv, 90):>15.1f}')
print(f'  p95  (reservoir)  : {np.percentile(rsv, 95):>15.1f}')
print(f'  p99  (reservoir)  : {np.percentile(rsv, 99):>15.1f}')
print(f'  等效页数 (@500tok) : {total_tokens/500:>15,.0f} 页')

print(f'\n── chunk_type ────────────────────────────────────────')
for k, v in sorted(by_chunk_type.items()):
    print(f'  {k:<15} chunks={v["chunks"]:>10,}  tokens={v["tokens"]:>12,}  '
          f'avg={v["tokens"]/v["chunks"]:.1f}')

print(f'\n── imrad_type ────────────────────────────────────────')
for k, v in sorted(by_imrad.items(), key=lambda x: -x[1]['tokens']):
    print(f'  {k:<15} chunks={v["chunks"]:>10,}  tokens={v["tokens"]:>12,}  '
          f'avg={v["tokens"]/v["chunks"]:.1f}')

print(f'\n── split_strategy ───────────────────────────────────')
for k, v in sorted(by_strategy.items(), key=lambda x: -x[1]['chunks']):
    print(f'  {k:<22} chunks={v["chunks"]:>10,}  tokens={v["tokens"]:>12,}')

print(f'\n── Token 分布直方图 ──────────────────────────────────')
max_cnt = max(hist.values())
for label, cnt in hist.items():
    bar = '█' * int(cnt / max_cnt * 35)
    print(f'  {label:<10}  {cnt:>10,}  {bar}')

## Cell 2 — 读取 Parquet 内容并输出为 JSON

In [ ]:
import json

# ── 配置 ─────────────────────────────────────────────────────────────────
OUTPUT_PATH  = Path(r'd:\Rag-Med\pipeline_output\chunks_sample.json')  # 输出路径
OUTPUT_JSONL = Path(r'd:\Rag-Med\pipeline_output\chunks_full.jsonl')    # 全量 JSONL 路径

# 模式：'sample'  → 取每个批次前 N 条，合并为单个 JSON 文件
#       'full'    → 全量流式写出为 JSONL（每行一条记录）
MODE         = 'sample'
SAMPLE_N     = 5           # sample 模式：每批取前几条
MAX_BATCHES  = 10          # sample 模式：最多读几个批次（None = 全部）

# 输出哪些字段（None = 全部）
EXPORT_COLS  = None   # 例：['chunk_id','text','chunk_type','imrad_type','token_count','journal','pub_year']

# ── 辅助：Int64 / NaN 序列化 ──────────────────────────────────────────────
def to_json_safe(record: dict) -> dict:
    out = {}
    for k, v in record.items():
        if v is None or (isinstance(v, float) and np.isnan(v)):
            out[k] = None
        elif hasattr(v, 'item'):   # numpy / pandas Int64
            out[k] = v.item()
        else:
            out[k] = v
    return out

# ── 执行 ──────────────────────────────────────────────────────────────────
target_files = batch_files if MAX_BATCHES is None else batch_files[:MAX_BATCHES]

if MODE == 'sample':
    # 每批取前 SAMPLE_N 条 → 合并 → 写单个 JSON 文件
    records = []
    for bf in target_files:
        df = pd.read_parquet(bf, columns=EXPORT_COLS)
        records.extend(df.head(SAMPLE_N).to_dict('records'))

    safe_records = [to_json_safe(r) for r in records]

    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        json.dump(safe_records, f, ensure_ascii=False, indent=2)

    print(f'已写出 {len(safe_records):,} 条记录 → {OUTPUT_PATH}')
    print(f'文件大小：{OUTPUT_PATH.stat().st_size / 1e6:.1f} MB')

    # 预览前 2 条
    print('\n预览（前 2 条）：')
    for r in safe_records[:2]:
        print(json.dumps(r, ensure_ascii=False, indent=2)[:600])
        print('...')

elif MODE == 'full':
    # 全量流式写出 JSONL（每行一条，适合超大数据集）
    total_written = 0
    with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
        for i, bf in enumerate(target_files):
            df = pd.read_parquet(bf, columns=EXPORT_COLS)
            for record in df.to_dict('records'):
                f.write(json.dumps(to_json_safe(record), ensure_ascii=False) + '\n')
            total_written += len(df)
            if (i + 1) % 100 == 0 or (i + 1) == len(target_files):
                print(f'  [{i+1}/{len(target_files)}]  写出 {total_written:,} 条')

    size_gb = OUTPUT_JSONL.stat().st_size / 1e9
    print(f'\n完成：{total_written:,} 条 → {OUTPUT_JSONL}  ({size_gb:.2f} GB)')

else:
    print(f'未知 MODE: {MODE!r}，请设置为 "sample" 或 "full"')

## Cell 3 — Baseline 数据集：每篇文章的 Token 数量

从 batch parquet 按 `doc_id` 聚合（每篇文章所有 chunk token 之和 ≈ 全文 token 数）。

In [ ]:
# ── 逐批聚合每篇文章的 token 总量 ────────────────────────────────────────
# 每篇文章的所有 chunk token 之和 ≈ 全文 token 数
# （不含被过滤的噪声节：参考文献、致谢等）

READ_COLS = ['doc_id', 'token_count']

doc_tokens: dict[str, int] = {}   # {doc_id: total_tokens}
LOG_EVERY  = 200

for i, bf in enumerate(batch_files):
    df = pd.read_parquet(bf, columns=READ_COLS).dropna()
    grp = df.groupby('doc_id')['token_count'].sum()
    for doc_id, tok in grp.items():
        doc_tokens[doc_id] = doc_tokens.get(doc_id, 0) + int(tok)
    if (i + 1) % LOG_EVERY == 0 or (i + 1) == len(batch_files):
        pct = (i + 1) / len(batch_files) * 100
        print(f'  [{i+1:>4}/{len(batch_files)}  {pct:.0f}%]  文献数={len(doc_tokens):,}')

# ── 转为 Series 计算统计 ─────────────────────────────────────────────────
tokens_per_doc = pd.Series(doc_tokens, dtype=int)

print(f'\n{"="*60}')
print(f'  Baseline 文章 Token 统计')
print(f'{"="*60}')
print(f'  文献总数          : {len(tokens_per_doc):>12,}')
print(f'  总 token 数       : {tokens_per_doc.sum():>12,}')
print(f'  均值              : {tokens_per_doc.mean():>12.1f}')
print(f'  中位数            : {tokens_per_doc.median():>12.1f}')
print(f'  p25               : {tokens_per_doc.quantile(0.25):>12.1f}')
print(f'  p75               : {tokens_per_doc.quantile(0.75):>12.1f}')
print(f'  p90               : {tokens_per_doc.quantile(0.90):>12.1f}')
print(f'  p95               : {tokens_per_doc.quantile(0.95):>12.1f}')
print(f'  p99               : {tokens_per_doc.quantile(0.99):>12.1f}')
print(f'  最大值            : {tokens_per_doc.max():>12,}')
print(f'  最小值            : {tokens_per_doc.min():>12,}')
print(f'  < 500  tok 文献   : {(tokens_per_doc < 500).sum():>12,}  '
      f'({(tokens_per_doc < 500).mean()*100:.1f}%)')
print(f'  500-2k tok 文献   : {((tokens_per_doc>=500)&(tokens_per_doc<2000)).sum():>12,}  '
      f'({((tokens_per_doc>=500)&(tokens_per_doc<2000)).mean()*100:.1f}%)')
print(f'  2k-8k  tok 文献   : {((tokens_per_doc>=2000)&(tokens_per_doc<8000)).sum():>12,}  '
      f'({((tokens_per_doc>=2000)&(tokens_per_doc<8000)).mean()*100:.1f}%)')
print(f'  > 8k   tok 文献   : {(tokens_per_doc >= 8000).sum():>12,}  '
      f'({(tokens_per_doc >= 8000).mean()*100:.1f}%)')

# ── Token 分布直方图（按文献维度）──────────────────────────────────────
BINS   = [0, 200, 500, 1000, 2000, 4000, 8000, 16000, 10**7]
LABELS = ['<200','200-499','500-999','1k-1.9k','2k-3.9k','4k-7.9k','8k-15.9k','>=16k']
print(f'\n── 文章 Token 分布直方图（共 {len(tokens_per_doc):,} 篇）──────────────')
max_cnt = 0
counts  = []
for j, label in enumerate(LABELS):
    lo, hi = BINS[j], BINS[j+1]
    cnt = int(((tokens_per_doc >= lo) & (tokens_per_doc < hi)).sum())
    counts.append((label, cnt))
    max_cnt = max(max_cnt, cnt)
for label, cnt in counts:
    bar = '█' * int(cnt / max_cnt * 35)
    print(f'  {label:<10}  {cnt:>8,}  {bar}')

# ── Top 10 最长文章 ──────────────────────────────────────────────────────
print(f'\n── 最长 10 篇文章 ───────────────────────────────────')
for doc_id, tok in tokens_per_doc.nlargest(10).items():
    print(f'  {doc_id:<20}  {tok:>8,} tokens')